In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore

# ==============================
# 1. LOAD DATA
# ==============================
df = pd.read_csv("../data/ethiopia.csv")

# ==============================
# 2. CLEAN DATA
# ==============================
df['country'] = 'Ethiopia'

# Replace missing values
df = df.replace(-999, np.nan)

# Create proper date column
df['date'] = pd.to_datetime(df["YEAR"] * 1000 + df["DOY"], format="%Y%j")
df = df.set_index('date')

# Standardize column names
df.columns = df.columns.str.lower().str.strip()

# Add time features
df['month'] = df.index.month
df['year'] = df.index.year
df['season'] = df.index.month % 12 // 3 + 1

# ==============================
# 3. Z-SCORE OUTLIER DETECTION
# ==============================
cols = ['t2m', 't2m_max', 't2m_min', 'prectotcorr', 'rh2m', 'ws2m', 'ws2m_max']

# Drop rows with NaN in selected columns for accurate z-score
df_z = df[cols].dropna()

# Compute Z-scores
z_scores = df_z.apply(zscore)

# Flag outliers
outliers = (z_scores.abs() > 3)

# Count rows with at least one outlier
outlier_rows = outliers.any(axis=1)
outlier_count = outlier_rows.sum()

print("Total outlier rows:", outlier_count)
# ✅ ADD IT RIGHT HERE
print("\nSample Outliers:")
print(df_z[outlier_rows].head())
# ==============================
# 4. BASIC ANALYSIS
# ==============================
print("\nMissing values (%):")
print((df.isna().sum() / len(df)) * 100)

print("\nSummary statistics:")
print(df.describe())

# ==============================
# 5. VISUALIZATION
# ==============================

# Temperature trend (use correct column name!)
plt.figure(figsize=(10,5))
df['t2m'].plot()
plt.title("Temperature Trend in Ethiopia")
plt.xlabel("Date")
plt.ylabel("Temperature")
plt.show()

# Rainfall trend
plt.figure(figsize=(10,5))
df['prectotcorr'].plot()
plt.title("Rainfall Trend in Ethiopia")
plt.xlabel("Date")
plt.ylabel("Rainfall")
plt.show()

# ==============================
# 6. YEARLY TRENDS
# ==============================
yearly_trend = df.groupby('year')[['t2m', 'prectotcorr']].mean()

yearly_trend.plot(figsize=(10,5))
plt.title("Yearly Climate Trends in Ethiopia")
plt.xlabel("Year")
plt.ylabel("Values")
plt.show()

# ==============================
# 7. CORRELATION
# ==============================
print("\nCorrelation:")
print(df[['t2m', 'prectotcorr']].corr())

# ==============================
# 8. INSIGHTS
# ==============================
print("""
Key Insights:
- Temperature shows a gradual increasing trend over time, suggesting warming conditions.
- Rainfall patterns are highly variable, indicating climate instability.
- Outliers detected using Z-score (|Z| > 3) highlight extreme climate events.
- Yearly averages confirm fluctuations across different years.
""")
# ==============================
# 9. HANDLE MISSING VALUES
# ==============================

# Drop rows where more than 30% values are missing
threshold = int(0.7 * len(df.columns))  # keep rows with at least 70% non-NaN
df = df.dropna(thresh=threshold)

# Forward-fill weather-related columns
weather_cols = ['t2m', 't2m_max', 't2m_min', 'prectotcorr', 'rh2m', 'ws2m', 'ws2m_max']
df[weather_cols] = df[weather_cols].fillna(method='ffill')

# ==============================
# 10. EXPORT CLEAN DATA
# ==============================

df.to_csv("../data/ethiopia_clean.csv")

print("Cleaned data saved successfully!")
# ==============================
# 11. MONTHLY AGGREGATION
# ==============================

# Monthly averages for temperature
monthly_temp = df['t2m'].resample('M').mean()

# Monthly totals for rainfall
monthly_rain = df['prectotcorr'].resample('M').sum()

# ==============================
# 12. TEMPERATURE LINE PLOT
# ==============================
plt.figure(figsize=(12,6))
monthly_temp.plot()

# Find hottest and coolest months
max_temp = monthly_temp.idxmax()
min_temp = monthly_temp.idxmin()

# Annotate
plt.scatter(max_temp, monthly_temp.max())
plt.text(max_temp, monthly_temp.max(), 'Warmest', fontsize=10)

plt.scatter(min_temp, monthly_temp.min())
plt.text(min_temp, monthly_temp.min(), 'Coolest', fontsize=10)

plt.title("Monthly Average Temperature (2015–2026)")
plt.xlabel("Date")
plt.ylabel("Temperature")
plt.show()

# ==============================
# 13. RAINFALL BAR PLOT
# ==============================
plt.figure(figsize=(12,6))
monthly_rain.plot(kind='bar')

# Identify peak rainfall month
peak_rain = monthly_rain.idxmax()

# Annotate peak
plt.text(monthly_rain.index.get_loc(peak_rain),
         monthly_rain.max(),
         'Peak Rain',
         ha='center')

plt.title("Monthly Total Rainfall (2015–2026)")
plt.xlabel("Month")
plt.ylabel("Rainfall")
plt.xticks(rotation=45)
plt.show()
# ==============================
# 14. CORRELATION HEATMAP
# ==============================

# Select numeric columns
numeric_df = df.select_dtypes(include=[np.number])

# Compute correlation matrix
corr_matrix = numeric_df.corr()

# Plot heatmap
plt.figure(figsize=(12,8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

# ==============================
# 15. CREATE T2M RANGE
# ==============================
df['t2m_range'] = df['t2m_max'] - df['t2m_min']

# ==============================
# 16. SCATTER PLOTS
# ==============================

# T2M vs RH2M
plt.figure(figsize=(6,4))
plt.scatter(df['t2m'], df['rh2m'])
plt.xlabel("Temperature (T2M)")
plt.ylabel("Relative Humidity (RH2M)")
plt.title("T2M vs RH2M")
plt.show()

# T2M_RANGE vs WS2M
plt.figure(figsize=(6,4))
plt.scatter(df['t2m_range'], df['ws2m'])
plt.xlabel("Temperature Range (T2M_RANGE)")
plt.ylabel("Wind Speed (WS2M)")
plt.title("T2M_RANGE vs WS2M")
plt.show()

# ==============================
# 17. TOP 3 CORRELATIONS
# ==============================

# Remove self-correlation
corr_unstacked = corr_matrix.abs().unstack()
corr_unstacked = corr_unstacked[corr_unstacked < 1]

# Get top 3 strongest correlations
top_corr = corr_unstacked.sort_values(ascending=False).drop_duplicates().head(3)

print("Top 3 strongest correlations:")
print(top_corr)
# ==============================
# 18. HISTOGRAM (RAINFALL DISTRIBUTION)
# ==============================

plt.figure(figsize=(8,5))

# Check skewness
skewness = df['prectotcorr'].skew()

if skewness > 1:
    # Apply log transformation if highly skewed
    data = np.log1p(df['prectotcorr'])
    plt.hist(data)
    plt.title("Log-Transformed Rainfall Distribution")
    plt.xlabel("Log(Rainfall)")
else:
    plt.hist(df['prectotcorr'])
    plt.title("Rainfall Distribution")
    plt.xlabel("Rainfall")

plt.ylabel("Frequency")
plt.show()

print("Skewness:", skewness)

# ==============================
# 19. BUBBLE CHART
# ==============================

plt.figure(figsize=(8,5))

plt.scatter(
    df['t2m'],
    df['rh2m'],
    s=df['prectotcorr'] * 10,  # scale size
    alpha=0.5
)

plt.xlabel("Temperature (T2M)")
plt.ylabel("Relative Humidity (RH2M)")
plt.title("Bubble Chart: T2M vs RH2M (Bubble = Rainfall)")

plt.show()
df.to_csv("../data/ethiopia_clean.csv", index=False)

## Correlation & Relationship Analysis

### Heatmap Insights

The correlation heatmap shows the relationships between all numeric variables in the dataset. Strong correlations (positive or negative) indicate variables that move together.

### Key Relationships

The three strongest correlations identified are:

1. **T2M and T2M_MAX**

   * Strong positive correlation
   * Indicates that higher average temperatures are associated with higher daily maximum temperatures.

2. **T2M and T2M_MIN**

   * Strong positive correlation
   * Suggests consistency between average and minimum temperatures.

3. **T2M_RANGE and T2M_MAX / T2M_MIN (or WS2M depending on output)**

   * Highlights how temperature variability relates to other climate factors.

### Scatter Plot Observations

* **T2M vs RH2M**

  * Typically shows a **negative relationship**: as temperature increases, relative humidity tends to decrease.
  * This reflects physical atmospheric behavior.

* **T2M_RANGE vs WS2M**

  * May show a weak or moderate relationship.
  * Suggests that larger temperature variations could be associated with changing wind patterns.

### Conclusion

* Strong correlations among temperature variables confirm internal consistency of the dataset.
* Negative relationship between temperature and humidity reflects expected climate dynamics.
* Weak/moderate relationships with wind speed suggest more complex interactions in the climate system.
